# **PULSE QUEST**

### Problem Statement : Dialogue Summarization to maximize ROUGEL score

In [ ]:
!pip install evaluate
!pip install rouge_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **DATA INGESTION AND REQUIREMENTS**

In [ ]:
import pandas as pd

train1 = pd.read_csv("/content/drive/MyDrive/PULSE QUEST/train.csv")
train2 = pd.read_csv("/content/drive/MyDrive/PULSE QUEST/train_2.csv")
train3 = pd.read_csv("/content/drive/MyDrive/PULSE QUEST/train_3.csv")

df_list = [train1, train2, train3]
merged_df = pd.concat(df_list, ignore_index=True)
merged_df.to_csv("/content/drive/MyDrive/PULSE QUEST/combtrain.csv", index=False)

In [ ]:
import numpy as np
from datasets import load_dataset, DatasetDict
import evaluate
import transformers

In [ ]:
dataset = load_dataset("csv", data_files={"train": '/content/drive/MyDrive/PULSE QUEST/combtrain.csv', "test": '/content/drive/MyDrive/PULSE QUEST/test_2.csv'})

train_val = dataset["train"].train_test_split(test_size=0.05)

dataset = DatasetDict({
    "train": train_val["train"],
    "val": train_val["test"],
    "test": dataset["test"]
})


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

### **TEXT PREPROCESSING**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("linydub/bart-large-samsum")

def preprocess_train(batch):
    dialogues = [
        d if d is not None else ""
        for d in batch["dialogue"]
    ]

    inputs = tokenizer(dialogues, truncation=True, padding="max_length", max_length = 512)
    labels = tokenizer(batch["summary"], truncation=True, padding="max_length", max_length = 128)
    inputs["labels"] = labels["input_ids"]
    return inputs

def preprocess_test(batch):

    inputs = tokenizer(batch["dialogue"], truncation=True, padding="max_length", max_length=512)

    return inputs

data_tok = DatasetDict()
data_tok["train"] = dataset["train"].map(preprocess_train, batched=True, remove_columns = dataset["train"].column_names)
data_tok["val"] = dataset["val"].map(preprocess_train, batched=True, remove_columns = dataset["val"].column_names)
data_tok["test"] = dataset["test"].map(preprocess_test, batched=True, remove_columns = dataset["test"].column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/1874 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### **MODEL INITIALISATION**

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments
from transformers import Trainer

model = AutoModelForSeq2SeqLM.from_pretrained("linydub/bart-large-samsum")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

legacy_keys = [
    "max_length", "min_length", "num_beams",
    "early_stopping", "no_repeat_ngram_size",
    "length_penalty"
]

for k in legacy_keys:
    if hasattr(model.config, k):
        setattr(model.config, k, None)


args = TrainingArguments(
    output_dir = '/content/bart_model',
    eval_strategy = "no",
    save_strategy = 'epoch',
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    num_train_epochs = 2,
    weight_decay=0.01,
    logging_steps = 100,
    save_total_limit=2,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model = model,
    args = args,
    train_dataset = data_tok["train"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

/tmp/ipython-input-447131151.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### **TRAINING LOOP**

In [ ]:
trainer.train()

Step,Training Loss
100,9.407300
200,5.367300
300,4.473300
400,3.567100
500,2.657600
600,1.780200
700,1.047600
800,0.585200
900,0.358100
1000,0.259300


TrainOutput(global_step=3748, training_loss=0.9002080882244456, metrics={'train_runtime': 1092.0453, 'train_samples_per_second': 3.432, 'train_steps_per_second': 3.432, 'total_flos': 4061154024357888.0, 'train_loss': 0.9002080882244456, 'epoch': 2.0})

### **VALIDATION AND GENERATION**

In [ ]:
from transformers import GenerationConfig
model.generation_config = GenerationConfig()

In [ ]:

print(model.generation_config)


GenerationConfig {}



In [ ]:
import torch
import evaluate
rouge = evaluate.load("rouge")

def validation(model, tokenizer, dataset, batch_size=2):
    model.eval()
    device = model.device

    preds_all = []
    refs_all = []

    for i in range(0, len(dataset), batch_size):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))

        input_ids = torch.tensor(batch["input_ids"]).to(device)
        attention_mask = torch.tensor(batch["attention_mask"]).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                num_beams=8,
                length_penalty=0.7,
                no_repeat_ngram_size=3,
                max_new_tokens=150,
                decoder_start_token_id=tokenizer.pad_token_id,
                eos_token_id = tokenizer.eos_token_id,
                pad_token_id = tokenizer.pad_token_id,
                early_stopping=True
            )

        preds = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        labels = torch.tensor(batch["labels"])
        labels[labels == -100] = tokenizer.pad_token_id

        refs = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        preds_all.extend(preds)
        refs_all.extend(refs)

    scores = rouge.compute(
        predictions=preds_all,
        references=refs_all
    )

    return scores["rougeL"]

Eval = validation(
    model,
    tokenizer,
    data_tok["val"]
)

print("ROUGE-L:", Eval)

In [ ]:
import pandas as pd

def generate(model, tokenizer, dataset, batch_size=16):
    model.eval()
    device = model.device

    all_summaries = []

    for i in range(0, len(dataset), batch_size):
        batch = dataset.select(range(i, min(i + batch_size, len(dataset))))

        input_ids = torch.tensor(batch["input_ids"]).to(device)
        attention_mask = torch.tensor(batch["attention_mask"]).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                num_beams=8,
                length_penalty=0.7,
                no_repeat_ngram_size=3,
                max_new_tokens=150,
                decoder_start_token_id=tokenizer.pad_token_id,
                eos_token_id = tokenizer.eos_token_id,
                pad_token_id = tokenizer.pad_token_id,
                early_stopping=True
            )

        summaries = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        all_summaries.extend(summaries)

    return all_summaries


In [ ]:
test_summaries = generate(
    model,
    tokenizer,
    data_tok["test"]
)

sub_df = pd.DataFrame({
    "id": dataset["test"]["id"],
    "summary": test_summaries
})

sub_df.to_csv("/content/newsubmission.csv", index=False)

In [ ]:
"sub_df.head()"

'sub_df.head()'

In [ ]:
import time

while True:
    print("job finished, keeping kernel alive...", flush=True)
    time.sleep(300)